This notebook asks how many phytoplankton groups the SDP pigments can tell apart inside the target eddies, following the hierarchical clustering of Kramer and Siegel (2019). Most diagnostic pigments occur in several groups, so a single pigment does not name a group, but pigments carried by the same group rise and fall together across samples. The clustering finds the sets of pigments that covary, and each set is one group the data can resolve.

Method:
- Each of the 12 accessory pigments is divided by Tchla, so the correlations describe composition rather than biomass. On absolute concentrations every pigment correlates with every other through Tchla.
- Pearson r between pigments, 1 - r as the distance, Ward's linkage, and the cophenetic correlation as the check that the dendrogram reproduces the distances.
- Flat clusters at the linkage cutoff of 1.0 that Kramer and Siegel used for the global data set.

Two correlation matrices:
- Across eddy-composites: the interior means of `gold/eddy_pigment_table.parquet`, one row per target eddy-composite that passed the 80% Rrs coverage rule.
- Within eddies: one correlation matrix per target eddy over its interior pixels with Tchla above zero, averaged over the eddies. Pooling the pixels of every eddy does not work, because a pixel with Tchla near zero has ratios in the thousands and Pearson collapses every pigment into one cluster.

Labels:
- Each leaf is a pigment, colored by its cluster.
- `pigment_groups` lists, for each pigment, every group in which Table 1 of Kramer and Siegel (2019, adapted from Jeffrey et al. 2011) marks it as often present, always present, or unique. A pigment can belong to several groups and a group carries several pigments, so the map is many-to-many; the table under the figure lists it per pigment.
- A cluster is labelled with the groups for which at least half of the carrying pigments sit in that cluster. A tie labels both clusters. The union of the members' groups would name almost every group for the middle cluster and say nothing.

Target eddies are the ones of `eddy_evolution.ipynb`: cyclones formed north of the Gulf Stream axis and ended south, or formed within `NEAR_AXIS_KM` (150 km) of the axis and ended south, and anticyclones under the reversed rule.

In [ ]:
from pathlib import Path
from typing import cast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator
from scipy.cluster.hierarchy import cophenet, dendrogram, fcluster, linkage, set_link_color_palette
from scipy.spatial.distance import squareform

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENT = 'gulf_stream_20240305_20260531'
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
identity_columns = ['polarity', 'track_id']
pigments = ['Tchla', 'Fuco', 'HexFuco', 'ButFuco', 'Perid', 'Chlc12', 'Chlc3', 'Allo', 'MV_chlb', 'Neo', 'Viola', 'Zea', 'DV_chla']
accessory = pigments[1:]
pigment_labels = {
    'Fuco': 'Fucoxanthin', 'HexFuco': "19'-Hex-fucoxanthin", 'ButFuco': "19'-But-fucoxanthin",
    'Perid': 'Peridinin', 'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin',
    'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
}
pigment_groups = {
    'Fuco': ['Diatoms', 'Dinoflagellates', 'Chrysophytes', 'Pelagophytes', 'Haptophytes'],
    'HexFuco': ['Haptophytes'],
    'ButFuco': ['Pelagophytes', 'Haptophytes'],
    'Perid': ['Dinoflagellates'],
    'Chlc12': ['Diatoms', 'Dinoflagellates', 'Chrysophytes', 'Pelagophytes', 'Haptophytes', 'Cryptophytes'],
    'Chlc3': ['Diatoms', 'Dinoflagellates', 'Haptophytes'],
    'Allo': ['Cryptophytes'],
    'MV_chlb': ['Green algae'],
    'Neo': ['Green algae'],
    'Viola': ['Chrysophytes', 'Green algae'],
    'Zea': ['Chrysophytes', 'Pelagophytes', 'Green algae', 'Cyanobacteria'],
    'DV_chla': ['Cyanobacteria'],
}
group_pigments = {group: [pigment for pigment in accessory if group in pigment_groups[pigment]] for group in dict.fromkeys(group for pigment in accessory for group in pigment_groups[pigment])}

eddy_tracks = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
pigment_table = pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet')
pigment_table['polarity'] = cast(pd.Series, pigment_table['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
pigment_table = pigment_table.rename(columns={f'eddy_mean_{pigment}': pigment for pigment in pigments})
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['crossed_axis'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['near_axis_birth'] = (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM)
    & eddy_tracks['death_side'].eq(target_class.str[1])
)
eddy_tracks['is_target'] = eddy_tracks['crossed_axis'] | eddy_tracks['near_axis_birth']
target_pigments = pigment_table.merge(eddy_tracks.loc[eddy_tracks['is_target'], identity_columns], on=identity_columns)
pixel_columns = {'T chla': 'Tchla', 'DV chla': 'DV_chla', 'MV chlb': 'MV_chlb', 'chl c1+c2': 'Chlc12', 'chl c3': 'Chlc3'}
pixels = pd.concat([
    pd.read_parquet(DATA_DIR / f'silver/pigments/{polarity}/eddy_{track_id}_pigments.parquet').assign(polarity=polarity)
    for polarity, track_id in target_pigments[identity_columns].drop_duplicates().itertuples(index=False)
], ignore_index=True).rename(columns=pixel_columns)
pixels = pixels.merge(target_pigments[identity_columns + ['date']], on=identity_columns + ['date'])
interior = pixels.loc[pixels['inside_contour'] & pixels['Tchla'].gt(0)]

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

display(cast(pd.DataFrame, interior.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'), interior_pixels=('Tchla', 'size'),
)).assign(composites=target_pigments.groupby('polarity').size()))

In [ ]:
correlations = {
    'Across eddy-composites': (cast(pd.DataFrame, target_pigments[accessory]).div(target_pigments['Tchla'], axis=0).corr().to_numpy(), len(target_pigments)),
    'Within eddies': (np.mean([member[accessory].div(member['Tchla'], axis=0).corr().to_numpy() for _, member in interior.groupby(identity_columns)], axis=0), interior.groupby(identity_columns).ngroups),
}
cutoff = 1.0
set_link_color_palette(['#1b9e77', '#d95f02', '#7570b3', '#e7298a'])
assignments = pd.DataFrame({'pigment': [pigment_labels[pigment] for pigment in accessory], 'literature_groups': [', '.join(pigment_groups[pigment]) for pigment in accessory]})
fit_rows = []
tree_axes = []
cluster_fig = cast(Figure, plt.figure(figsize=(6.69, 4.9)))
outer = cluster_fig.add_gridspec(1, 2, left=0.075, right=0.99, bottom=0.29, top=0.94, wspace=0.12)
for column, (letter, (title, (corr, n))) in enumerate(zip('ab', correlations.items())):
    distance = squareform((2 - corr - corr.T) / 2, checks=False)
    Z = linkage(distance, method='ward')
    fit_rows.append({'correlation': title, 'n': n, 'cophenetic_r': cophenet(Z, distance)[0]})
    assignments[title] = fcluster(Z, t=cutoff, criterion='distance')
    tree_ax = cast(Axes, cluster_fig.add_subplot(outer[column], sharey=tree_axes[0] if tree_axes else None))
    tree_axes.append(tree_ax)
    tree = dendrogram(Z, ax=tree_ax, labels=[pigment_labels[pigment] for pigment in accessory], leaf_rotation=90, leaf_font_size=8, color_threshold=cutoff, above_threshold_color='#444444')
    for label, color in zip(tree_ax.xaxis.get_ticklabels(), tree['leaves_color_list']):
        label.set_color(color)
    for color in dict.fromkeys(tree['leaves_color_list']):
        members = [position for position, leaf_color in enumerate(tree['leaves_color_list']) if leaf_color == color]
        cluster_pigments = {accessory[tree['leaves'][position]] for position in members}
        cluster_groups = [group for group, carriers in group_pigments.items() if 2 * len(cluster_pigments.intersection(carriers)) >= len(carriers)]
        tree_ax.text(10 * np.mean(members) + 5, 0.98, ',\n'.join(cluster_groups), transform=tree_ax.get_xaxis_transform(), ha='center', va='top', color=color)
    tree_ax.axhline(cutoff, color='#999999', linewidth=0.6, linestyle=(0, (4, 2.5)))  # pyright: ignore[reportArgumentType]
    tree_ax.tick_params(axis='x', length=0)
    tree_ax.tick_params(axis='y', labelleft=column == 0)
    tree_ax.spines['bottom'].set_visible(False)
    tree_ax.set_title(f'$\\bf{{({letter})}}$ {title}', loc='left')
tree_axes[0].autoscale(axis='y')
ticks = cast(np.ndarray, MaxNLocator(nbins=4, steps=[1, 2, 5, 10]).tick_values(0, tree_axes[0].get_ylim()[1]))
ticks = ticks[ticks <= tree_axes[0].get_ylim()[1]]
tree_axes[0].set_ylim(0, 1.5 * tree_axes[0].get_ylim()[1])
tree_axes[0].yaxis.set_ticks(ticks)
for tree_ax in tree_axes:
    tree_ax.spines['left'].set_bounds(0, ticks[-1])
tree_axes[0].set_ylabel('Linkage distance')
plt.show()
display(pd.DataFrame(fit_rows).round(3))
display(assignments.sort_values(list(correlations), ignore_index=True))